# Improving wPINNs on the moving shock

This notebook distills the experiments in this repo into a single **optimal recipe** and
measures the improvement over the baseline wPINN on the Burgers moving shock
($u_0 = 1$ for $x\le 0$, $0$ for $x>0$; domain $x\in[-1,1]$, $t\in[0,0.45]$).

**The optimal recipe = baseline wPINN + two changes:**
1. **Hard maximum-principle bounds** — the solution network output is passed through a
   scaled sigmoid so $u$ is *forced* into the physical band $[\min u_0,\max u_0]=[0,1]$.
   This enforces a real physical law for free (no competing loss term).
2. **Residual-adaptive collocation sampling** — every 250 epochs the collocation points are
   moved toward where the PDE residual $|u_t+uu_x|$ is largest (the shock), so the jump is
   properly resolved.

Both runs use **identical** architecture, seed, optimizer and training budget (3000 epochs),
so the difference in error is attributable to the recipe.

Models are produced by `train_optimal.py`:
```
MODE=baseline python3 train_optimal.py   # -> ShockWave/baseline
MODE=optimal  python3 train_optimal.py   # -> ShockWave/optimal
```

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from godunov import exact_solution, relative_l1_error

CASE = "Moving"
MODELS = {
    "baseline": "ShockWave/baseline/ModelSol.pkl",
    "optimal":  "ShockWave/optimal/ModelSol.pkl",
}

def load(path):
    m = torch.load(path, map_location="cpu", weights_only=False)
    m.eval()
    return m

def pred(model, x, t):
    inp = torch.tensor(np.column_stack([np.full_like(x, t), x]), dtype=torch.float32)
    with torch.no_grad():
        return model(inp).numpy().reshape(-1)

models = {k: load(p) for k, p in MODELS.items()}
x = np.linspace(-1, 1, 1000)
print("loaded:", list(models))

## Headline: how much did the recipe improve accuracy?

Global relative $L^1$ error against the exact entropy solution, averaged over the whole
space-time domain.

In [ ]:
ts = np.linspace(0.01, 0.45, 45)

def global_rel_l1(model):
    num = den = 0.0
    for t in ts:
        ue = exact_solution(CASE, x, t)
        up = pred(model, x, t)
        num += np.sum(np.abs(up - ue))
        den += np.sum(np.abs(ue))
    return num / den

err = {k: global_rel_l1(m) for k, m in models.items()}
improvement = 100.0 * (err["baseline"] - err["optimal"]) / err["baseline"]
factor = err["baseline"] / err["optimal"]

print(f"baseline   relative L1 = {err['baseline']:.4f}")
print(f"optimal    relative L1 = {err['optimal']:.4f}")
print(f"\nimprovement = {improvement:.1f}%   ({factor:.1f}x lower error)")

## Solution slices: baseline vs optimal vs exact

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True)
fig.suptitle("Moving shock — baseline vs optimal (hard bounds + adaptive sampling)", fontsize=13)
for ax, t in zip(axes, [0.10, 0.25, 0.45]):
    ax.axhline(0, color="0.85", lw=1)
    ax.axhline(1, color="0.85", lw=1)
    ax.plot(x, exact_solution(CASE, x, t), "k-", lw=2.2, label="Exact")
    ax.plot(x, pred(models["baseline"], x, t), "r--", lw=1.5, label="baseline")
    ax.plot(x, pred(models["optimal"],  x, t), "g-",  lw=1.6, label="optimal")
    ax.set_title(f"t = {t}")
    ax.set_xlabel("x")
    ax.grid(True, ls=":")
    ax.legend(fontsize=8)
axes[0].set_ylabel("u")
plt.tight_layout()
plt.savefig("optimal_wpinn_slices.png", dpi=150)
plt.show()

## Error vs time

The baseline error grows as the shock sharpens; the optimal recipe keeps it low and flat.

In [ ]:
tt = np.linspace(0.02, 0.45, 40)
plt.figure(figsize=(7, 4))
for k, style in [("baseline", "r--"), ("optimal", "g-")]:
    e = [relative_l1_error(pred(models[k], x, t), exact_solution(CASE, x, t)) for t in tt]
    plt.plot(tt, e, style, label=k)
plt.xlabel("t"); plt.ylabel("relative L1")
plt.title("Moving shock — relative L1 error vs time")
plt.legend(); plt.grid(True, ls=":")
plt.tight_layout()
plt.savefig("optimal_wpinn_error_vs_time.png", dpi=150)
plt.show()

## Physical bounds

The baseline overshoots/undershoots the physical band $[0,1]$; the optimal model stays inside it
by construction.

In [ ]:
print(f"{'variant':10s} {'min u':>9} {'max u':>9}   (physical band [0, 1])")
for k, m in models.items():
    gmin = min(pred(m, x, t).min() for t in ts)
    gmax = max(pred(m, x, t).max() for t in ts)
    print(f"{k:10s} {gmin:+9.4f} {gmax:+9.4f}")

## Summary

On the moving shock, combining a **maximum-principle output constraint** with
**residual-adaptive collocation sampling** reduces the relative $L^1$ error substantially
versus the baseline wPINN (same architecture, seed and training budget), while keeping the
solution inside the physical band $[0,1]$ — see the printed improvement above.

Caveat established by the other experiments in this repo: the adaptive-sampling half of the
recipe is a **shock-specific** tool — it helps shock-dominated problems but hurts smooth
solutions (rarefaction, sine-bump), where uniform coverage is better. The hard-bounds half is
universally safe.